# 6.4 分布式协同：每架无人机独立思考

> **AirSim 配置**：使用 `1-6-settings.json`（2架无人机）。如已在运行笔记本3，需重启 AirSim 重置无人机位置。

## 核心思想

分布式协同的核心区别：**没有中央指挥官**，每架无人机拥有自己的 LLM "大脑"，通过一个**共享消息板**交换信息、协调行动。

```
Drone1 [自己的LLM] ──读/写──→ 共享消息板 ←──读/写── [自己的LLM] Drone2
```

每架无人机的决策流程：
1. 读取消息板上其他无人机的状态
2. 结合自己的任务目标，让 LLM 做出决策
3. 执行飞行动作
4. 将结果发布到消息板

这模拟了真实场景中无人机通过无线通信交换信息的过程。

## Step 1：导入工具

In [1]:
import json
from airsim_tools import connect_airsim, takeoff, fly_to, get_state, call_llm

## Step 2：连接 AirSim

In [2]:
client = connect_airsim()
client.reset()

Connected!
Client Ver:1 (Min Req: 1), Server Ver:1 (Min Req: 1)

已连接到 AirSim 模拟器


## Step 3：定义共享消息板和无人机大脑

**消息板**是一个简单的 Python 列表，所有无人机都可以读写。

**drone_brain** 是每架无人机的"大脑"函数——它读取消息板、调LLM决策、执行动作、发布结果。

In [4]:
# ==================== 分布式协同核心代码 ====================

# 共享消息板（模拟无线通信）
message_board = []


def drone_brain(client, drone_id, mission, message_board):
    """
    单架无人机的"大脑"——独立思考、独立行动。
    
    Args:
        client: AirSim 客户端
        drone_id: 本机名称，如 'Drone1'
        mission: 总体任务描述
        message_board: 共享消息板（列表，读写其他无人机的状态）
    """
    print(f"\n{'='*50}")
    print(f"[{drone_id}] 开始独立思考...")
    
    # 1. 读取消息板：获取其他无人机的信息
    others_info = [m for m in message_board if m["from"] != drone_id]
    if others_info:
        print(f"[{drone_id}] 从消息板读取到: {others_info}")
    else:
        print(f"[{drone_id}] 消息板为空，我是第一个行动的")
    
    # 2. LLM 独立决策
    decision_prompt = f"""你是无人机 {drone_id}，正在执行分布式协同任务。

总体任务：{mission}

其他无人机的状态（来自消息板）：
{json.dumps(others_info, ensure_ascii=False) if others_info else "暂无（你是第一个行动的）"}

请根据以上信息，决定你应该飞往哪个坐标。
要求：
- 不要和其他无人机去同一个位置
- 输出纯JSON，格式：{{"target": [x, y, z], "reason": "简短理由"}}
- z坐标为负表示高度（如-10表示10米高）
- 不要输出JSON以外的任何文字
"""
    
    decision_text = call_llm(decision_prompt)
    print(f"[{drone_id}] LLM决策: {decision_text}")
    
    decision = json.loads(decision_text)
    target = decision["target"]
    reason = decision.get("reason", "")
    
    # 3. 执行飞行
    print(f"[{drone_id}] 决定飞往 {target}，理由: {reason}")
    takeoff(client, drone_id)
    fly_to(client, drone_id, target[0], target[1], target[2])
    
    # 4. 获取实际位置
    actual_pos = get_state(client, drone_id)
    
    # 5. 发布到消息板
    message = {
        "from": drone_id,
        "target": target,
        "actual_pos": actual_pos,
        "status": "已到达",
        "reason": reason
    }
    message_board.append(message)
    print(f"[{drone_id}] 已发布到消息板: {message}")
    
    return message

## Step 4：执行分布式协同

两架无人机依次执行（模拟分布式场景）。关键观察点：**Drone2 能看到 Drone1 的消息，从而做出不同的决策。**

In [5]:
# 总体任务
mission = "搜索区域：需要侦察坐标(-10,10,-10)附近和(10,10,-10)附近两个区域，两架无人机分工协作，不要去同一个地方。"

# 清空消息板
message_board = []

# Drone1 先行动（此时消息板为空）
drone_brain(client, "Drone1", mission, message_board)

# Drone2 后行动（此时能看到 Drone1 的消息）
drone_brain(client, "Drone2", mission, message_board)


[Drone1] 开始独立思考...
[Drone1] 消息板为空，我是第一个行动的
[Drone1] LLM决策: {"target": [-10, 10, -10], "reason": "作为首个行动的无人机，选择该区域执行侦察任务，预留另一侦察区域给其余无人机，避免任务重叠"}
[Drone1] 决定飞往 [-10, 10, -10]，理由: 作为首个行动的无人机，选择该区域执行侦察任务，预留另一侦察区域给其余无人机，避免任务重叠
[Drone1] 起飞完成
[Drone1] 已到达 (-10, 10, -10)
[Drone1] 已发布到消息板: {'from': 'Drone1', 'target': [-10, 10, -10], 'actual_pos': {'x': -10.12, 'y': 10.13, 'z': -10.02}, 'status': '已到达', 'reason': '作为首个行动的无人机，选择该区域执行侦察任务，预留另一侦察区域给其余无人机，避免任务重叠'}

[Drone2] 开始独立思考...
[Drone2] 从消息板读取到: [{'from': 'Drone1', 'target': [-10, 10, -10], 'actual_pos': {'x': -10.12, 'y': 10.13, 'z': -10.02}, 'status': '已到达', 'reason': '作为首个行动的无人机，选择该区域执行侦察任务，预留另一侦察区域给其余无人机，避免任务重叠'}]
[Drone2] LLM决策: {"target": [10, 10, -10], "reason": "Drone1已负责(-10,10,-10)区域侦察，为避免任务重叠，选择剩余的(10,10,-10)区域执行侦察任务"}
[Drone2] 决定飞往 [10, 10, -10]，理由: Drone1已负责(-10,10,-10)区域侦察，为避免任务重叠，选择剩余的(10,10,-10)区域执行侦察任务
[Drone2] 起飞完成
[Drone2] 已到达 (10, 10, -10)
[Drone2] 已发布到消息板: {'from': 'Drone2', 'target': [10, 10, -10], 'actual_pos': {'x': 1

{'from': 'Drone2',
 'target': [10, 10, -10],
 'actual_pos': {'x': 10.1, 'y': 10.1, 'z': -10.0},
 'status': '已到达',
 'reason': 'Drone1已负责(-10,10,-10)区域侦察，为避免任务重叠，选择剩余的(10,10,-10)区域执行侦察任务'}

## Step 5：查看最终状态

In [6]:
print("\n===== 消息板最终状态 =====")
for msg in message_board:
    print(f"  {msg['from']}: 飞往{msg['target']}，理由: {msg['reason']}")

print("\n===== 各无人机实际位置 =====")
for drone_id in ["Drone1", "Drone2"]:
    pos = get_state(client, drone_id)
    print(f"  {drone_id}: {pos}")


===== 消息板最终状态 =====
  Drone1: 飞往[-10, 10, -10]，理由: 作为首个行动的无人机，选择该区域执行侦察任务，预留另一侦察区域给其余无人机，避免任务重叠
  Drone2: 飞往[10, 10, -10]，理由: Drone1已负责(-10,10,-10)区域侦察，为避免任务重叠，选择剩余的(10,10,-10)区域执行侦察任务

===== 各无人机实际位置 =====
  Drone1: {'x': -10.24, 'y': 10.25, 'z': -10.09}
  Drone2: {'x': 10.22, 'y': 10.22, 'z': -10.07}


## Step 6：LLM 协同评估

让 LLM 评估两架无人机的协同效果。

In [7]:
eval_prompt = f"""请评估以下分布式无人机协同任务的执行质量：

任务：{mission}
消息板记录：{json.dumps(message_board, ensure_ascii=False)}

请分析：
1. 两架无人机是否成功分工（去了不同位置）？
2. Drone2是否利用了Drone1的信息做出协调决策？
3. 整体任务完成度如何？
"""

evaluation = call_llm(eval_prompt)
print("===== 协同评估 =====")
print(evaluation)

===== 协同评估 =====
1. 已成功分工：两架无人机分别前往两个不同的指定侦察区域，无任务重叠，完全符合分工要求。
2. 是，Drone2明确参考了Drone1已认领(-10,10,-10)区域的公开信息，主动选择剩余的(10,10,-10)区域执行任务，属于利用同伴信息做出的协调决策。
3. 整体任务完成度为100%：两个待侦察区域均被覆盖，两架无人机均抵达对应目标点的合理误差范围内，协作逻辑顺畅无冲突，完全达成所有预设任务要求。


## 流程总结

分布式协同的核心机制：

```
                    ┌──────────────────┐
                    │   共享消息板       │
                    │ (模拟无线通信)     │
                    └───┬──────────┬───┘
                  读/写 │          │ 读/写
              ┌────────┘          └────────┐
              ▼                             ▼
     ┌──────────────┐             ┌──────────────┐
     │   Drone1     │             │   Drone2     │
     │ ┌──────────┐ │             │ ┌──────────┐ │
     │ │ LLM大脑  │ │             │ │ LLM大脑  │ │
     │ └──────────┘ │             │ └──────────┘ │
     │  独立决策     │             │  独立决策     │
     │  独立执行     │             │  独立执行     │
     └──────────────┘             └──────────────┘
```

**与中心化的关键区别**：

| 对比 | 中心化（上一节） | 分布式（本节） |
|------|-----------------|---------------|
| 决策者 | 1个指挥官LLM | 每架无人机各自的LLM |
| 通信 | 指挥官→工作者（单向） | 消息板（双向共享） |
| Drone2的信息 | 由指挥官分配 | 自己从消息板读取Drone1的结果 |
| 适应性 | 低（预先规划好） | 高（可根据实时信息调整） |

**核心观察**：Drone2 在决策时能看到 Drone1 已经去了某个位置，因此会自动选择去另一个位置——这就是分布式协同的魅力。

---

## 高级扩展方向

在此基础上，可以进一步探索：

- **多轮协同**：让无人机执行多轮任务，每轮都根据消息板更新决策
- **动态任务调整**：某架无人机发现目标后，通过消息板通知其他无人机改变任务
- **冲突消解**：当两架无人机选择了相同目标时，通过消息板协商
- **Agent框架集成**：理解了原理后，可以用 LangGraph 等框架实现更复杂的工作流

理解了这两种基本模式（中心化 + 分布式），就掌握了多无人机Agent协同的核心思想。